# ERASE Holistic Unlearning Cost — Operator-Level FLOPs

**Paper**: *Fast Exact Unlearning for In-Context Learning Data for LLMs*  
(Muresanu et al., ICML 2025)  
OpenReview: https://openreview.net/forum?id=3ZesmvOJ6o

---

## What this notebook does

Computes the **Holistic Unlearning Cost** (Definition 4.1) for **ERASE-{1,2,3,4}-shot**,  
using `torch.utils.flop_counter.FlopCounterMode` for **accurate, operator-level FLOP counts**.

$$C(M) = \frac{U_M - U_{1\text{-SISA}}}{I_{1\text{-SISA}} - I_M}$$

- $U_M$ = unlearning FLOP cost per request for method $M$
- $I_M$ = inference FLOP cost per query for method $M$
- Higher $C(M)$ = better (M can handle more inferences per unlearning request before
  becoming more expensive than 1-SISA)

**Key ERASE insight**: ERASE replaces fine-tuning with In-Context Learning (ICL).  
Unlearning cost ≈ 0 (just update a k-means index, O(d)), but inference is more  
expensive (k examples added to every prompt).

**Platform**: Lightning AI (GPU recommended)

---

### How to swap model / data
All configurable values are in the `CONFIG` dict (Cell 3).  
Look for `# ═══ SWAP: MODEL ═══` and `# ═══ SWAP: DATA ═══` comments.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Cell 2: Install Dependencies
#  Lightning AI has PyTorch pre-installed; we add the rest.
# ═══════════════════════════════════════════════════════════════════

!pip install -q transformers datasets sentence-transformers accelerate peft

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Cell 3: Imports & Configuration
#
#  ALL tuneable parameters live here.  Change CONFIG values to swap
#  model, dataset, ERASE hyper-parameters, or profiling settings.
# ═══════════════════════════════════════════════════════════════════

import json
import time
import copy
import warnings
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm import tqdm

# ─────────────────────────────────────────────────────────────────
#  CONFIG — Edit these values to customise the experiment
# ─────────────────────────────────────────────────────────────────
CONFIG = {
    # ═══ SWAP: MODEL ═══════════════════════════════════════════
    # Replace model_name with any HuggingFace causal-LM.
    # For gated models (e.g. LLaMA-2), set hf_token.
    # Examples:
    #   "meta-llama/Llama-2-7b-chat-hf"  (needs token + Meta approval)
    #   "mistralai/Mistral-7B-Instruct-v0.2"
    #   "microsoft/phi-2"
    "model_name":   "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "hf_token":     None,
    "torch_dtype":  "float16",      # "float16" | "bfloat16"

    # ═══ SWAP: DATA ════════════════════════════════════════════
    # Replace with any HuggingFace text dataset.
    # Requirements: must have a text column (set text_column below).
    "dataset_name":   "swj0419/WikiMIA",
    "dataset_split":  "WikiMIA_length128",
    "text_column":    "input",        # column containing document text
    "n_docs":         50,             # total documents to use
    "max_seq_length": 128,            # tokeniser truncation length

    # ─── ERASE parameters (paper defaults) ────────────────────
    "epsilon":         0.05,          # quantization granularity
    "max_kmeans_iter": 10,            # k-means iterations
    "encoder_name":    "sentence-transformers/all-MiniLM-L6-v2",
    "k_shot_list":     [1, 2, 3, 4],  # ERASE variants to evaluate

    # ─── FLOP profiling ───────────────────────────────────────
    "n_profile_steps": 5,             # training steps to profile
    "profile_batch_size": 1,          # batch size for profiling
    "n_epochs":        1,             # training epochs (for scaling)

    # ─── Chat format (for ICL prompts) ────────────────────────
    # "tinyllama" | "mistral" | "plain"
    "chat_format":     "tinyllama",
}

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Cell 4: Load Model
#
#  ═══ SWAP: MODEL ═══
#  To use a different LLM, change CONFIG["model_name"] in Cell 3.
#  For gated models, also set CONFIG["hf_token"].
# ═══════════════════════════════════════════════════════════════════

from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"Loading model: {CONFIG['model_name']} ...")

tokenizer = AutoTokenizer.from_pretrained(
    CONFIG["model_name"],
    token=CONFIG["hf_token"],
    trust_remote_code=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"],
    token=CONFIG["hf_token"],
    torch_dtype=getattr(torch, CONFIG["torch_dtype"]),
    device_map="auto",
    trust_remote_code=True,
)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"\n{'='*60}")
print(f"  Model loaded: {CONFIG['model_name']}")
print(f"  Parameters:   {n_params:,}")
print(f"  Dtype:        {CONFIG['torch_dtype']}")
print(f"  Device:       {next(model.parameters()).device}")
print(f"{'='*60}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Cell 5: Load Data
#
#  ═══ SWAP: DATA ═══
#  To use a different dataset, change CONFIG["dataset_name"],
#  CONFIG["dataset_split"], and CONFIG["text_column"] in Cell 3.
#
#  The only requirement is a text column with document strings.
# ═══════════════════════════════════════════════════════════════════

from datasets import load_dataset

print(f"Loading dataset: {CONFIG['dataset_name']} / {CONFIG['dataset_split']} ...")

raw_dataset = load_dataset(
    CONFIG["dataset_name"],
    split=CONFIG["dataset_split"],
)

# ─── Extract text documents ───────────────────────────────────
# SWAP: adjust this block if your dataset has a different structure
all_texts = raw_dataset[CONFIG["text_column"]][:CONFIG["n_docs"]]

# Use a sample query for inference profiling
# SWAP: replace with a representative query for your domain
sample_query = "What is this document about?"

print(f"\n{'='*60}")
print(f"  Dataset:           {CONFIG['dataset_name']}")
print(f"  Split:             {CONFIG['dataset_split']}")
print(f"  Documents loaded:  {len(all_texts)}")
print(f"  Sample (100 chars): {all_texts[0][:100]}...")
print(f"{'='*60}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Cell 6: ERASE Core Classes (stripped from erase_unlearning.py)
#
#  This cell defines the ERASE algorithm inline — no external .py
#  file dependency.  SISA classes have been removed; this notebook
#  is standalone for the ERASE method only.
#
#  Classes defined:
#    - UnlearningResult  (dataclass)
#    - QuantizedKMeans   (Ginart et al. 2019)
#    - ERASE             (Algorithm 1 from Muresanu et al. 2025)
#    - HolisticCostEvaluator  (Definition 4.1)
# ═══════════════════════════════════════════════════════════════════

from sentence_transformers import SentenceTransformer


# ── Data Class ──────────────────────────────────────────────────

@dataclass
class UnlearningResult:
    """
    Complete record of one exact unlearning operation.
    
    Fields
    ------
    idx               : original index in the dataset that was unlearned
    cluster_id        : which cluster the point belonged to
    refit_needed      : True -> full k-means refit was required (expensive path)
    centroid_changed  : True -> the quantized centroid changed
    was_selected      : True -> this point was the chosen ICL example
    new_selected_text : replacement ICL example (when was_selected=True)
    unlearning_flops  : estimated FLOPS for this operation
    wall_time_s       : wall-clock seconds
    """
    idx: int
    cluster_id: int
    refit_needed: bool
    centroid_changed: bool
    was_selected: bool
    new_selected_text: Optional[str]
    unlearning_flops: int
    wall_time_s: float

    def __str__(self) -> str:
        path = "REFIT" if self.refit_needed else (
            "SWAP" if self.was_selected else "FREE"
        )
        return (
            f"UnlearningResult(idx={self.idx}, cluster={self.cluster_id}, "
            f"path={path}, flops={self.unlearning_flops:,}, "
            f"time={self.wall_time_s*1000:.1f}ms)"
        )


# ── Quantized K-Means (Ginart et al. 2019) ─────────────────────

class QuantizedKMeans:
    """
    Quantized K-Means with dataset-size-independent exact unlearning.
    
    Core idea: snap centroids to an epsilon-lattice so that removing
    a single point usually does NOT change the quantized centroid.
    When it doesn't change -> O(d) unlearning (cheap path).
    When it does change   -> full refit, but P(change) = O(1/(eps*|D|)).
    """

    def __init__(self, n_clusters, epsilon=0.05, max_iter=10, random_state=42):
        self.n_clusters   = n_clusters
        self.epsilon      = epsilon
        self.max_iter     = max_iter
        self.random_state = random_state

        self.data_               = None  # (n_active, d)
        self.original_indices_   = None  # row -> original idx
        self.raw_centroids_      = None  # (k, d)
        self.quantized_centroids_ = None # (k, d)
        self.labels_             = None  # (n_active,)
        self.theta_              = None  # lattice offset, sampled ONCE
        self._rng = np.random.default_rng(random_state)

    def fit(self, X, original_indices=None):
        """Fit quantized k-means to embedding matrix X  (n, d)."""
        n, d = X.shape
        if n < self.n_clusters:
            raise ValueError(f"Need >= {self.n_clusters} samples, got {n}.")

        self.data_ = X.copy().astype(np.float64)
        self.original_indices_ = (
            np.arange(n) if original_indices is None
            else np.asarray(original_indices, dtype=int)
        )

        if self.theta_ is None:
            self.theta_ = self._rng.uniform(
                -self.epsilon / 2, self.epsilon / 2, d
            )

        self.raw_centroids_ = self._kmeanspp_init(self.data_)
        self.quantized_centroids_ = None

        for _ in range(self.max_iter):
            q_c = np.array([self._quantize(c) for c in self.raw_centroids_])
            dists = self._pairwise_l2(self.data_, q_c)
            new_labels = np.argmin(dists, axis=1)

            new_raw = np.zeros_like(self.raw_centroids_)
            for k in range(self.n_clusters):
                mask = new_labels == k
                if mask.sum() > 0:
                    new_raw[k] = self.data_[mask].mean(axis=0)
                else:
                    new_raw[k] = self.data_[
                        dists.min(axis=1).argmax()
                    ]

            new_q = np.array([self._quantize(c) for c in new_raw])
            if (
                self.quantized_centroids_ is not None
                and np.allclose(new_q, self.quantized_centroids_, atol=1e-12)
            ):
                self.labels_ = new_labels
                self.raw_centroids_ = new_raw
                break

            self.labels_ = new_labels
            self.raw_centroids_ = new_raw
            self.quantized_centroids_ = new_q

        self.quantized_centroids_ = np.array(
            [self._quantize(c) for c in self.raw_centroids_]
        )
        return self

    def unlearn(self, original_idx):
        """
        Remove one data point and decide whether a refit is needed.
        Returns (centroid_changed, refit_needed).
        """
        rows = np.where(self.original_indices_ == original_idx)[0]
        if len(rows) == 0:
            raise ValueError(f"original_idx={original_idx} not found.")
        row = int(rows[0])
        cluster_id = int(self.labels_[row])
        n_c = int((self.labels_ == cluster_id).sum())

        if n_c <= 1:
            self._remove_row(row)
            if len(self.data_) >= self.n_clusters:
                self.fit(self.data_, self.original_indices_)
            return True, True

        x_i = self.data_[row]
        new_raw = (n_c * self.raw_centroids_[cluster_id] - x_i) / (n_c - 1)
        new_q   = self._quantize(new_raw)
        centroid_changed = not np.allclose(
            new_q, self.quantized_centroids_[cluster_id], atol=1e-12
        )

        self._remove_row(row)

        if not centroid_changed:
            self.raw_centroids_[cluster_id] = new_raw
            return False, False
        else:
            if len(self.data_) >= self.n_clusters:
                self.fit(self.data_, self.original_indices_)
            return True, True

    def get_sorted_cluster(self, cluster_id):
        """Members of cluster_id sorted by distance to centroid."""
        mask = self.labels_ == cluster_id
        rows = np.where(mask)[0]
        if len(rows) == 0:
            return []
        centroid = self.quantized_centroids_[cluster_id]
        dists = np.linalg.norm(self.data_[rows] - centroid, axis=1)
        order = np.argsort(dists)
        return [
            (int(self.original_indices_[rows[i]]), float(dists[i]))
            for i in order
        ]

    def cluster_of(self, original_idx):
        rows = np.where(self.original_indices_ == original_idx)[0]
        if len(rows) == 0:
            raise ValueError(f"original_idx={original_idx} not found.")
        return int(self.labels_[rows[0]])

    @property
    def n_active(self):
        return len(self.data_) if self.data_ is not None else 0

    # ── Private helpers ───────────────────────────────────────
    def _quantize(self, centroid):
        return (
            np.round((centroid - self.theta_) / self.epsilon) * self.epsilon
            + self.theta_
        )

    def _kmeanspp_init(self, X):
        n = len(X)
        idx = int(self._rng.integers(0, n))
        centers = [X[idx].copy()]
        for _ in range(1, self.n_clusters):
            dists = np.array([
                min(np.linalg.norm(x - c) ** 2 for c in centers)
                for x in X
            ])
            probs = dists / dists.sum()
            idx = int(self._rng.choice(n, p=probs))
            centers.append(X[idx].copy())
        return np.array(centers)

    @staticmethod
    def _pairwise_l2(A, B):
        A2 = (A ** 2).sum(axis=1, keepdims=True)
        B2 = (B ** 2).sum(axis=1, keepdims=True).T
        AB = A @ B.T
        return np.sqrt(np.clip(A2 + B2 - 2 * AB, 0.0, None))

    def _remove_row(self, row):
        keep = np.ones(len(self.data_), dtype=bool)
        keep[row] = False
        self.data_             = self.data_[keep]
        self.original_indices_ = self.original_indices_[keep]
        self.labels_           = self.labels_[keep]


# ── ERASE Algorithm (Algorithm 1) ──────────────────────────────

class ERASE:
    """
    ERASE: Efficient Removal And Selection of Examples.
    
    1. Encode examples with Sentence-BERT.
    2. Run quantized k-means (k = n_examples).
    3. Select the closest example to each centroid as the ICL example.
    
    On unlearning:
      - FREE: point was not selected, centroid unchanged -> O(d)
      - SWAP: point was selected, centroid unchanged -> pick next closest, O(d)
      - REFIT: centroid changed -> full refit, O(n*k*d*iter), rare
    """

    def __init__(
        self,
        n_examples=4,
        epsilon=0.05,
        encoder_name="sentence-transformers/all-MiniLM-L6-v2",
        max_iter=10,
        random_state=42,
    ):
        self.n_examples   = n_examples
        self.epsilon      = epsilon
        self.encoder_name = encoder_name
        self.max_iter     = max_iter
        self.random_state = random_state

        self._encoder = None
        self._qkm = QuantizedKMeans(
            n_clusters=n_examples,
            epsilon=epsilon,
            max_iter=max_iter,
            random_state=random_state,
        )

        self.texts_     = []
        self.metadata_  = []
        self.embeddings_ = None
        self.selected_  = {}   # cluster_id -> original_idx
        self._log       = []

    def fit(self, texts, metadata=None, precomputed_embeddings=None):
        """
        Encode texts and cluster with quantized k-means.
        
        Parameters
        ----------
        texts : list of str
        metadata : optional list of dicts
        precomputed_embeddings : (n, d) array — skip encoding if provided
        """
        if len(texts) < self.n_examples:
            raise ValueError(
                f"Need >= {self.n_examples} texts, got {len(texts)}."
            )
        self.texts_    = list(texts)
        self.metadata_ = metadata or [{} for _ in texts]

        if precomputed_embeddings is not None:
            self.embeddings_ = np.asarray(
                precomputed_embeddings, dtype=np.float64
            )
            print(f"[ERASE] Using {len(texts)} precomputed embeddings "
                  f"(d={self.embeddings_.shape[1]}).")
        else:
            if self._encoder is None:
                print(f"[ERASE] Loading encoder: {self.encoder_name}")
                self._encoder = SentenceTransformer(self.encoder_name)
            print(f"[ERASE] Encoding {len(texts)} examples...")
            self.embeddings_ = self._encoder.encode(
                texts, show_progress_bar=True, convert_to_numpy=True
            ).astype(np.float64)

        print(f"[ERASE] Running quantized k-means "
              f"(k={self.n_examples}, eps={self.epsilon})...")
        self._qkm.fit(
            self.embeddings_,
            original_indices=np.arange(len(texts)),
        )
        self._refresh_selected()
        print(f"[ERASE] Ready. Selected {len(self.selected_)} examples.")
        return self

    def get_selected_examples(self):
        """Return the k currently selected ICL examples."""
        out = []
        for cid in sorted(self.selected_):
            orig_idx = self.selected_[cid]
            text = self.texts_[orig_idx]
            if not text:
                continue
            sm = self._qkm.get_sorted_cluster(cid)
            dist = next((d for i, d in sm if i == orig_idx), float("nan"))
            out.append({
                "idx": orig_idx, "text": text,
                "cluster": cid, "distance_to_centroid": dist,
                "metadata": self.metadata_[orig_idx],
            })
        return out

    def get_icl_prompt(
        self, query, system_msg="Answer using only the context provided.",
        chat_format="tinyllama",
    ):
        """Build a few-shot ICL prompt with the selected examples."""
        examples = self.get_selected_examples()
        few_shot = "\n".join(f"Example: {ex['text']}" for ex in examples)
        if chat_format == "tinyllama":
            return (
                f"<|system|>{system_msg}</s>"
                f"<|user|>{few_shot}\n\nQuestion: {query}</s>"
                f"<|assistant|>"
            )
        elif chat_format == "mistral":
            return (
                f"[INST] {system_msg}\n\n"
                f"{few_shot}\n\nQuestion: {query} [/INST]"
            )
        else:
            return f"{system_msg}\n\n{few_shot}\n\nQuestion: {query}"

    def unlearn(self, original_idx):
        """
        Exactly unlearn the example at original_idx.
        Returns an UnlearningResult.
        """
        t0 = time.perf_counter()
        if original_idx >= len(self.texts_) or not self.texts_[original_idx]:
            raise ValueError(f"original_idx={original_idx} invalid.")

        cluster_id   = self._qkm.cluster_of(original_idx)
        was_selected = (self.selected_.get(cluster_id) == original_idx)
        centroid_changed, refit_needed = self._qkm.unlearn(original_idx)
        self.texts_[original_idx] = ""

        new_selected_text = None
        if refit_needed:
            self._refresh_selected()
        elif was_selected and not centroid_changed:
            sm = self._qkm.get_sorted_cluster(cluster_id)
            valid = [(i, d) for i, d in sm if self.texts_[i]]
            if valid:
                new_idx = valid[0][0]
                self.selected_[cluster_id] = new_idx
                new_selected_text = self.texts_[new_idx]
            else:
                self._qkm.fit(self._qkm.data_, self._qkm.original_indices_)
                self._refresh_selected()
                refit_needed = True

        d = self.embeddings_.shape[1]
        n_active = sum(1 for t in self.texts_ if t)
        if not refit_needed:
            flops = 3 * d
        else:
            flops = n_active * self.n_examples * d * self.max_iter * 2

        result = UnlearningResult(
            idx=original_idx, cluster_id=cluster_id,
            refit_needed=refit_needed,
            centroid_changed=centroid_changed,
            was_selected=was_selected,
            new_selected_text=new_selected_text,
            unlearning_flops=flops,
            wall_time_s=time.perf_counter() - t0,
        )
        self._log.append(result)
        return result

    def _refresh_selected(self):
        self.selected_ = {}
        for cid in range(self.n_examples):
            sm = self._qkm.get_sorted_cluster(cid)
            valid = [(i, d) for i, d in sm if self.texts_[i]]
            if valid:
                self.selected_[cid] = valid[0][0]


# ── Holistic Cost Evaluator (Definition 4.1) ──────────────────

class HolisticCostEvaluator:
    """
    Definition 4.1: C(M) = (U_M - U_1SISA) / (I_1SISA - I_M)
    
    Higher C(M) = better.  M can handle C(M) inferences per
    unlearning request before becoming more expensive than 1-SISA.
    """

    def __init__(self):
        self._methods = {}

    def add_method(self, name, U, I):
        """Register a method with its U (unlearning) and I (inference) FLOPs."""
        self._methods[name] = {"U": float(U), "I": float(I)}

    def compute_C(self, U_M, I_M, U_sisa, I_sisa):
        """Compute holistic cost C(M) vs 1-SISA."""
        denom = I_sisa - I_M
        if abs(denom) < 1.0:
            return float("inf")
        return (U_M - U_sisa) / denom

    def compare_all(self):
        """Compare all methods vs '1-SISA' baseline. Returns DataFrame."""
        if "1-SISA" not in self._methods:
            raise ValueError("Must register '1-SISA' first.")
        sisa = self._methods["1-SISA"]
        rows = []
        for name, costs in self._methods.items():
            C = self.compute_C(costs["U"], costs["I"], sisa["U"], sisa["I"])
            rows.append({
                "method": name,
                "unlearning_flops": costs["U"],
                "inference_flops":  costs["I"],
                "holistic_C":       C,
                "better_than_1SISA": (C > 1.0 or np.isinf(C)),
            })
        return (
            pd.DataFrame(rows)
            .sort_values("holistic_C", ascending=False)
            .reset_index(drop=True)
        )

    def print_report(self):
        df = self.compare_all()
        print("\n" + "=" * 76)
        print("  HOLISTIC UNLEARNING COST  (Def 4.1 — Muresanu et al. 2025)")
        print("  Higher C(M) = better.  C(M) = inferences per unlearning")
        print("  request before method M becomes >= 1-SISA cost.")
        print("=" * 76)
        print(f"  {'Method':<22} {'C(M)':>12}  {'U (FLOPs)':>14}  "
              f"{'I (FLOPs)':>14}  OK?")
        print("  " + "-" * 72)
        for _, row in df.iterrows():
            c_str = (f"{row['holistic_C']:,.0f}"
                     if not np.isinf(row['holistic_C']) else "inf")
            ok = "YES" if row["better_than_1SISA"] else "NO"
            print(f"  {row['method']:<22} {c_str:>12}  "
                  f"{row['unlearning_flops']:>14.2e}  "
                  f"{row['inference_flops']:>14.2e}  {ok}")
        print("=" * 76 + "\n")


print("[OK] ERASE core classes defined.")

---
## Operator-Level FLOP Profiling

We use `torch.utils.flop_counter.FlopCounterMode` to capture **actual ATen operator FLOPs**  
for both training (forward + backward) and inference (forward only).

This replaces the analytical `6·N·T` estimate with measured values.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Cell 8: Profile 1-SISA Baseline
#
#  ERASE's holistic cost is defined *relative* to the 1-SISA
#  baseline, so we need both:
#    U_1SISA = total training FLOPS / 2  (checkpoint assumption)
#    I_1SISA = inference FLOPs for a plain query (no ICL context)
#
#  We profile N training steps (forward + backward), then scale
#  to the full dataset.  This is accurate because per-step FLOPs
#  are deterministic for fixed-shape batches.
# ═══════════════════════════════════════════════════════════════════

from torch.utils.flop_counter import FlopCounterMode

print("="*60)
print("  STEP 1: Profile 1-SISA TRAINING cost")
print("="*60)

# ── Prepare a profiling batch ─────────────────────────────────
profile_batch_enc = tokenizer(
    all_texts[:CONFIG["profile_batch_size"]],
    padding="max_length",
    truncation=True,
    max_length=CONFIG["max_seq_length"],
    return_tensors="pt",
)
profile_batch = {k: v.to(device) for k, v in profile_batch_enc.items()}
profile_batch["labels"] = profile_batch["input_ids"].clone()

# ── Enable gradients on ALL parameters for full fine-tuning ──
# (This matches the paper's SISA cost model: full retraining)
for p in model.parameters():
    p.requires_grad_(True)
model.train()

# ── Profile N training steps (forward + backward) ────────────
train_step_flops_list = []
print(f"\nProfiling {CONFIG['n_profile_steps']} training steps...")

for step_i in range(CONFIG["n_profile_steps"]):
    # Fresh counter for each step
    flop_counter = FlopCounterMode(display=False)
    with flop_counter:
        outputs = model(**profile_batch)
        loss = outputs.loss
        loss.backward()

    step_flops = flop_counter.get_total_flops()
    train_step_flops_list.append(step_flops)
    print(f"  Step {step_i+1}: loss={loss.item():.4f}  "
          f"FLOPs={step_flops:,.0f}")

    # Zero grads (no optimizer step — we only measure FLOPs)
    model.zero_grad(set_to_none=True)

# ── Disable gradients, return to eval mode ───────────────────
for p in model.parameters():
    p.requires_grad_(False)
model.eval()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

# ── Scale to full dataset ─────────────────────────────────────
avg_train_step_flops = float(np.mean(train_step_flops_list))
total_train_samples  = len(all_texts)
total_train_steps    = (
    total_train_samples / CONFIG["profile_batch_size"]
) * CONFIG["n_epochs"]
total_training_flops = avg_train_step_flops * total_train_steps
U_1SISA = total_training_flops / 2  # checkpoint assumption

print(f"\n--- 1-SISA Training Cost ---")
print(f"  Avg FLOPs/step:        {avg_train_step_flops:>18,.0f}")
print(f"  Total steps (scaled):  {total_train_steps:>18,.0f}")
print(f"  Total training FLOPs:  {total_training_flops:>18,.0f}")
print(f"  U_1SISA (half):        {U_1SISA:>18,.0f}")

# ══════════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print("  STEP 2: Profile 1-SISA INFERENCE cost")
print("="*60)

# 1-SISA inference = fine-tuned model on plain query (no ICL)
query_enc = tokenizer(
    sample_query,
    return_tensors="pt",
    truncation=True,
    max_length=CONFIG["max_seq_length"],
)
query_inputs = {k: v.to(device) for k, v in query_enc.items()}

flop_counter = FlopCounterMode(display=False)
with flop_counter:
    with torch.no_grad():
        _ = model(**query_inputs)

I_1SISA = float(flop_counter.get_total_flops())

print(f"  Query: \"{sample_query}\"")
print(f"  Query length:    {query_inputs['input_ids'].shape[1]} tokens")
print(f"  I_1SISA FLOPs:   {I_1SISA:>18,.0f}")

print(f"\n{'='*60}")
print(f"  1-SISA Baseline Summary")
print(f"  U_1SISA = {U_1SISA:.4e}")
print(f"  I_1SISA = {I_1SISA:.4e}")
print(f"{'='*60}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Cell 9: Profile ERASE k-shot Variants
#
#  For each k in {1, 2, 3, 4}:
#    1. Fit ERASE with k clusters
#    2. Build ICL prompt with k examples + query
#    3. Profile inference FLOPs -> I_ERASE_k
#    4. Measure ERASE unlearning cost -> U_ERASE_k (expected ~0)
# ═══════════════════════════════════════════════════════════════════

print("="*60)
print("  Profiling ERASE k-shot Variants")
print("="*60)

erase_results = {}  # k -> {"U": ..., "I": ..., "prompt_tokens": ...}

for k in CONFIG["k_shot_list"]:
    print(f"\n--- ERASE-{k}shot ---")

    # ── 1. Fit ERASE ──────────────────────────────────────────
    erase_k = ERASE(
        n_examples=k,
        epsilon=CONFIG["epsilon"],
        encoder_name=CONFIG["encoder_name"],
        max_iter=CONFIG["max_kmeans_iter"],
    )
    erase_k.fit(all_texts)

    # ── 2. Build ICL prompt ───────────────────────────────────
    icl_prompt = erase_k.get_icl_prompt(
        sample_query,
        chat_format=CONFIG["chat_format"],
    )

    # ── 3. Profile inference FLOPs ────────────────────────────
    icl_enc = tokenizer(
        icl_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024,  # ICL prompts can be long
    )
    icl_inputs = {k_: v.to(device) for k_, v in icl_enc.items()}

    flop_counter = FlopCounterMode(display=False)
    with flop_counter:
        with torch.no_grad():
            _ = model(**icl_inputs)

    I_erase_k = float(flop_counter.get_total_flops())
    prompt_tokens = icl_inputs["input_ids"].shape[1]

    # ── 4. Measure unlearning cost ────────────────────────────
    # ERASE unlearning is O(d) expected — practically zero
    # compared to SISA retraining.  Profile the actual operation:
    selected_examples = erase_k.get_selected_examples()
    if selected_examples:
        test_idx = selected_examples[0]["idx"]
        ul_result = erase_k.unlearn(test_idx)
        U_erase_k = float(ul_result.unlearning_flops)
    else:
        U_erase_k = 0.0

    erase_results[k] = {
        "U": U_erase_k,
        "I": I_erase_k,
        "prompt_tokens": prompt_tokens,
        "unlearn_path": str(ul_result) if selected_examples else "N/A",
    }

    print(f"  Prompt tokens:     {prompt_tokens}")
    print(f"  I_ERASE_{k}:       {I_erase_k:>18,.0f} FLOPs")
    print(f"  U_ERASE_{k}:       {U_erase_k:>18,.0f} FLOPs")
    if selected_examples:
        print(f"  Unlearn result:    {ul_result}")

print(f"\n{'='*60}")
print("  ERASE Profiling Complete")
print(f"{'='*60}")

---
## Holistic Cost Computation

$$C(M) = \frac{U_M - U_{1\text{-SISA}}}{I_{1\text{-SISA}} - I_M}$$

For ERASE: $U_M \approx 0$, $I_M > I_{1\text{-SISA}}$ (longer prompt),  
so $C = U_{1\text{-SISA}} / (I_M - I_{1\text{-SISA}})$ — always positive and large.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Cell 10: Compute Holistic Cost Table
#
#  Assembles all profiled values and computes C(M) for each
#  ERASE-{k}shot variant against the 1-SISA baseline.
# ═══════════════════════════════════════════════════════════════════

evaluator = HolisticCostEvaluator()

# Register 1-SISA baseline
evaluator.add_method("1-SISA", U=U_1SISA, I=I_1SISA)

# Register ERASE variants
for k, res in erase_results.items():
    evaluator.add_method(f"ERASE-{k}shot", U=res["U"], I=res["I"])

# Print the report
evaluator.print_report()

# Get the full DataFrame
df_holistic = evaluator.compare_all()
print("\nFull results table:")
print(df_holistic.to_string(index=False))

# ── Interpretation ────────────────────────────────────────────
print("\n" + "="*60)
print("  INTERPRETATION")
print("="*60)
for _, row in df_holistic.iterrows():
    if row["method"] == "1-SISA":
        continue
    c = row['holistic_C']
    if np.isinf(c):
        print(f"  {row['method']}: Always cheaper than 1-SISA (C=inf)")
    elif c > 0:
        print(f"  {row['method']}: Cheaper than 1-SISA if you do < {c:,.0f}"
              f" inferences per unlearning request")
    else:
        print(f"  {row['method']}: More expensive than 1-SISA (C={c:,.0f})")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Cell 11: Export Results
#
#  Save profiled FLOPs and holistic costs to JSON and CSV for
#  reproducibility and further analysis.
# ═══════════════════════════════════════════════════════════════════

export = {
    "experiment": "ERASE Holistic Cost (Operator-Level FLOPs)",
    "paper": "Muresanu et al., ICML 2025",
    "model": CONFIG["model_name"],
    "dataset": CONFIG["dataset_name"],
    "n_docs": CONFIG["n_docs"],
    "n_profile_steps": CONFIG["n_profile_steps"],
    "profiler": "torch.utils.flop_counter.FlopCounterMode",
    "baseline": {
        "method": "1-SISA",
        "U_flops": U_1SISA,
        "I_flops": I_1SISA,
        "avg_train_step_flops": avg_train_step_flops,
        "total_train_steps_scaled": total_train_steps,
    },
    "erase_variants": {},
    "holistic_costs": {},
}

for k, res in erase_results.items():
    export["erase_variants"][f"ERASE-{k}shot"] = {
        "U_flops": res["U"],
        "I_flops": res["I"],
        "prompt_tokens": res["prompt_tokens"],
    }

for _, row in df_holistic.iterrows():
    c_val = row['holistic_C']
    export["holistic_costs"][row["method"]] = (
        "inf" if np.isinf(c_val) else float(c_val)
    )

# ── Save JSON ─────────────────────────────────────────────────
out_json = Path("erase_holistic_results.json")
with open(out_json, "w") as f:
    json.dump(export, f, indent=2, default=str)
print(f"Results saved to: {out_json.resolve()}")

# ── Save CSV ──────────────────────────────────────────────────
out_csv = Path("erase_holistic_results.csv")
df_holistic.to_csv(out_csv, index=False)
print(f"Table saved to:   {out_csv.resolve()}")

# ── Print final summary ───────────────────────────────────────
print("\n" + "="*60)
print("  ERASE HOLISTIC COST EXPERIMENT COMPLETE")
print("  Model: " + CONFIG["model_name"])
print("  Data:  " + CONFIG["dataset_name"])
print("="*60)